# Data Preparation
Merge `hotel.csv` and `distance2coast.csv` on `hotel_id` to produce `hotel_with_distance.csv`.

In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")

## Load & Inspect

In [2]:
hotels = pd.read_csv(DATA_DIR / "hotel.csv", encoding="utf-8-sig")
dist = pd.read_csv(DATA_DIR / "distance2coast.csv")

print("hotel.csv      :", hotels.shape)
print("distance2coast :", dist.shape)
hotels.head(2)

hotel.csv      : (8574, 41)
distance2coast : (29446, 6)


,hotel_id,chain_id,chain_name,brand_id,brand_name,hotel_name,hotel_formerly_name,hotel_translated_name,addressline1,addressline2,...,rates_from,continent_id,continent_name,city_id,country_id,number_of_reviews,rating_average,rates_currency,rates_from_exclusive,accommodation_type
0,163,0,No chain,0,NaN,Ramana Saigon Hotel,Amara Hotel,Ramana Saigon Hotel,"323 Le Van Sy Street, District 3",NaN,...,NaN,2,Asia,13170,38,1713,8.1,USD,NaN,Hotel
1,902,0,No chain,0,NaN,Lotte Hotel Saigon,Legend Hotel Saigon,Lotte Hotel Saigon,"2A-4A Ton Duc Thang Street, District 1",NaN,...,NaN,2,Asia,13170,38,6260,9.0,USD,NaN,Hotel


In [3]:
dist.head(2)

,hotel_id,longitude,latitude,hotel_coordinate,distance2coastline,nearest_coordinate
0,163,106.678101,10.787597,POINT (106.678101 10.787597),47.465,POINT (11907922.454514029 1165509.8719933222)
1,902,106.706797,10.778689,POINT (106.706797 10.778689),44.573,POINT (11907922.454514029 1165509.8719933222)


## Deduplication Check

hotel.csv      — duplicate hotel_ids: 0
distance2coast — duplicate hotel_ids: 0


## Pre-merge Cleanup

Drop `longitude` and `latitude` from `distance2coast` — they duplicate columns already present in `hotel.csv`.

In [5]:
dist_clean = dist.drop(columns=["longitude", "latitude"])
print("Columns added from distance2coast:", dist_clean.columns.tolist())

Columns added from distance2coast: ['hotel_id', 'hotel_coordinate', 'distance2coastline', 'nearest_coordinate']


## Merge

Left join on `hotel_id`: keep all 8 574 hotels from `hotel.csv`, attach distance data where available.

In [6]:
merged = hotels.merge(dist_clean, on="hotel_id", how="left")
print("Merged shape:", merged.shape)
merged.head(3)

Merged shape: (8574, 44)


,hotel_id,chain_id,chain_name,brand_id,brand_name,hotel_name,hotel_formerly_name,hotel_translated_name,addressline1,addressline2,...,city_id,country_id,number_of_reviews,rating_average,rates_currency,rates_from_exclusive,accommodation_type,hotel_coordinate,distance2coastline,nearest_coordinate
0,163,0,No chain,0,NaN,Ramana Saigon Hotel,Amara Hotel,Ramana Saigon Hotel,"323 Le Van Sy Street, District 3",NaN,...,13170,38,1713,8.1,USD,NaN,Hotel,POINT (106.678101 10.787597),47.465,POINT (11907922.454514029 1165509.8719933222)
1,902,0,No chain,0,NaN,Lotte Hotel Saigon,Legend Hotel Saigon,Lotte Hotel Saigon,"2A-4A Ton Duc Thang Street, District 1",NaN,...,13170,38,6260,9.0,USD,NaN,Hotel,POINT (106.706797 10.778689),44.573,POINT (11907922.454514029 1165509.8719933222)
2,3274,27,Pan Pacific Hotels and Resorts,1004,PARKROYAL Hotels & Resorts,PARKROYAL Saigon,Parkroyal Saigon Hotel,PARKROYAL Saigon,"309B-311 Nguyen Van Troi Street, Tan Binh Dist",NaN,...,13170,38,986,8.2,USD,NaN,Hotel,POINT (106.668122 10.799623),49.212,POINT (11907922.454514029 1165509.8719933222)


## Validation

In [7]:
assert merged.shape[0] == hotels.shape[0], "Row count changed after merge — check for duplicates"

missing_dist = merged['distance2coastline'].isna().sum()
total = len(merged)
print(f"Hotels with distance data : {total - missing_dist:,} / {total:,}")
print(f"Hotels missing distance   : {missing_dist:,}")

merged[['hotel_id', 'hotel_name', 'distance2coastline', 'hotel_coordinate', 'nearest_coordinate']].head(5)

Hotels with distance data : 8,574 / 8,574
Hotels missing distance   : 0


,hotel_id,hotel_name,distance2coastline,hotel_coordinate,nearest_coordinate
0,163,Ramana Saigon Hotel,47.465,POINT (106.678101 10.787597),POINT (11907922.454514029 1165509.8719933222)
1,902,Lotte Hotel Saigon,44.573,POINT (106.706797 10.778689),POINT (11907922.454514029 1165509.8719933222)
2,3274,PARKROYAL Saigon,49.212,POINT (106.668122 10.799623),POINT (11907922.454514029 1165509.8719933222)
3,9195,Renaissance Riverside Hotel Saigon,44.256,POINT (106.706283 10.774478),POINT (11907922.454514029 1165509.8719933222)
4,9698,TTC Hotel Dalat,72.746,POINT (108.437042 11.942183),POINT (12134860.5102322 1295168.6063270567)


## Export

In [8]:
out_path = DATA_DIR / "hotel_with_distance.csv"
merged.to_csv(out_path,encoding='utf-8-sig', index=False)
print(f"Saved → {out_path}")
print(f"Shape : {merged.shape}")
print(f"Columns ({len(merged.columns)}): {merged.columns.tolist()}")

Saved → ..\data\hotel_with_distance.csv
Shape : (8574, 44)
Columns (44): ['hotel_id', 'chain_id', 'chain_name', 'brand_id', 'brand_name', 'hotel_name', 'hotel_formerly_name', 'hotel_translated_name', 'addressline1', 'addressline2', 'zipcode', 'city', 'state', 'country', 'countryisocode', 'star_rating', 'longitude', 'latitude', 'url', 'checkin', 'checkout', 'numberrooms', 'numberfloors', 'yearopened', 'yearrenovated', 'photo1', 'photo2', 'photo3', 'photo4', 'photo5', 'overview', 'rates_from', 'continent_id', 'continent_name', 'city_id', 'country_id', 'number_of_reviews', 'rating_average', 'rates_currency', 'rates_from_exclusive', 'accommodation_type', 'hotel_coordinate', 'distance2coastline', 'nearest_coordinate']


In [16]:
choosen = merged[merged['number_of_reviews'] > 500]
choosen.shape

(1150, 44)

In [17]:
choosen[choosen.distance2coastline <= 0.5]

,hotel_id,chain_id,chain_name,brand_id,brand_name,hotel_name,hotel_formerly_name,hotel_translated_name,addressline1,addressline2,...,city_id,country_id,number_of_reviews,rating_average,rates_currency,rates_from_exclusive,accommodation_type,hotel_coordinate,distance2coastline,nearest_coordinate
40,10996,0,No chain,894,No brand,Hoi An Beach Resort,NaN,Hoi An Beach Resort,01 Cua Dai Road,NaN,...,16552,38,2253,8.5,USD,NaN,"Hotel, Resort",POINT (108.367332 15.896595),0.126,POINT (12063464.805837113 1781159.9328711773)
45,11005,0,No chain,0,NaN,Nha Trang Lodge Hotel,NaN,Nha Trang Lodge Hotel,42 Tran Phu Street,NaN,...,2679,38,744,7.9,USD,NaN,"Hotel, Lodge",POINT (109.196205 12.241484),0.107,POINT (12155558.873486977 1364141.9679585632)
47,11007,0,No chain,0,NaN,Yasaka Saigon Resort Hotel & Spa,NaN,Yasaka Saigon Resort Hotel & Spa,18 Tran Phu Street,NaN,...,2679,38,1513,7.4,USD,NaN,Hotel,POINT (109.196066 12.249085),0.131,POINT (12155519.616244497 1365007.4803484965)
48,11015,0,No chain,894,No brand,Palace Hotel,NaN,Palace Hotel,01 Nguyen Trai street,NaN,...,17190,38,1019,8.1,USD,NaN,Hotel,POINT (107.075928 10.342858),0.358,POINT (11919279.991180802 1150002.1420214414)
49,11016,0,No chain,0,NaN,Petro House Hotel,NaN,Petro House Hotel,"63 Tran Hung Dao st., Ward 1",NaN,...,17190,38,660,7.6,USD,NaN,Hotel,POINT (107.075018 10.349212),0.435,POINT (11919174.949243547 1150473.0241608808)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6341,37875166,0,No chain,894,No brand,Mandala Cham Bay Mui Ne,NaN,Mandala Cham Bay Mui Ne,ĐT716 Street,NaN,...,16264,38,622,7.9,USD,NaN,Hotel,POINT (108.344748 11.004217),0.303,POINT (12061166.951352747 1224331.7503520523)
6374,37960522,0,No chain,894,No brand,Mangata Beachfront Hotel,NaN,Mangata Beachfront Hotel,108 Võ Nguyên Giáp,NaN,...,16440,38,501,9.0,USD,NaN,Hotel,POINT (108.24572 16.078787),0.028,POINT (12049830.577670198 1802024.6207490382)
6584,38797641,0,No chain,894,No brand,Pansy Beach Dana,NaN,Pansy Beach Dana,"33 34 Do ba ,My An , Ngu Hanh Son, Da nang",NaN,...,16440,38,914,9.3,USD,NaN,Hotel,POINT (108.245613 16.051613),0.246,POINT (12050084.32185825 1798955.7394773255)
6755,40283156,0,No chain,0,NaN,Golden Lotus Luxury Hotel Da Nang,NaN,Golden Lotus Luxury Hotel Da Nang,"198B Tran Bach Dang Street, Bac My Phu Ward, N...",NaN,...,16440,38,846,9.3,USD,NaN,Hotel,POINT (108.246615 16.052538),0.111,POINT (12050065.249890467 1799027.521723336)


In [18]:
out_path = DATA_DIR / "hotel_filtered.csv"
choosen.to_csv(out_path,encoding='utf-8-sig', index=False)